In [1]:
import asyncio
import uuid

In [2]:
from azure.identity.aio import DefaultAzureCredential

In [3]:
TASK = '''

# Develop a shopping cart

## Description

Develop a shopping cart. The shopping cart should allow users to add and remove products.

## Description

### 1. Using Python

### 2. Develop the shopping cart. You can use the following steps as a guide:

    - Create an API Rest with the following methods:

    - Get a list of beers using page offsets and limits.

    - Get beer details by id.

- Search for beer by name, description, tagline, food pairings, and price.

- Create a list of products on the main page.

- Create a search bar to filter products.

- Jump to the description page when the user clicks on a product.

- (Optional) Slicer to filter products by price.

- Create a shopping cart.

    Add products to the cart.

    Remove products from the cart.

    Calculate the total price of the products in the cart.

'''

In [4]:
import os

In [5]:
from semantic_kernel.agents import AgentGroupChat, AzureAIAgent, AzureAIAgentSettings, AzureAIAgentThread
from semantic_kernel.agents.strategies import TerminationStrategy
from semantic_kernel.contents import AuthorRole
from azure.ai.projects.models import MessageRole

In [6]:
class ApprovalTerminationStrategy(TerminationStrategy):
    """A strategy for determining when an agent should terminate."""

    async def should_agent_terminate(self, agent, history):
        """Check if the agent should terminate."""
        return "saved" in history[-1].content.lower()

In [ ]:
arch_agent_id = "Your Arch Agent ID
code_agent_id = "Your Code Agent ID"

In [8]:
ai_agent_settings = AzureAIAgentSettings.create()

In [9]:
async with (
     DefaultAzureCredential() as creds,
     AzureAIAgent.create_client(credential=creds) as client,
):  

    arch_agent_definition = await client.agents.get_agent(arch_agent_id)
    code_agent_definition = await client.agents.get_agent(code_agent_id)


    agent_arch = AzureAIAgent(
            client=client,
            definition=arch_agent_definition,
    )

    agent_code = AzureAIAgent(
            client=client,
            definition=code_agent_definition,
    )


    chat = AgentGroupChat(
            agents=[agent_arch, agent_code],
            termination_strategy=ApprovalTerminationStrategy(agents=[agent_code], maximum_iterations=10),
    )

    thread: AzureAIAgentThread = None

    try:
            # 6. Add the task as a message to the group chat
            await chat.add_chat_message(message=TASK)
            print(f"# {AuthorRole.USER}: '{TASK}'")
            # 7. Invoke the chat
            async for content in chat.invoke():
                print(f"# {content.role} - {content.name or '*'}: '{content.content}'")
                if content.name == "DevOpsGenCode":
                        if len(content.items) > 1:
                              file_name = f"{uuid.uuid4()}.zip"
                              await client.agents.save_file(file_id=content.items[1].file_id, file_name=file_name,target_dir="./arch")
            
    finally:
            # 8. Cleanup: Delete the agents
            await chat.reset()

# AuthorRole.USER: '

# Develop a shopping cart

## Description

Develop a shopping cart. The shopping cart should allow users to add and remove products.

## Description

### 1. Using Python

### 2. Develop the shopping cart. You can use the following steps as a guide:

    - Create an API Rest with the following methods:

    - Get a list of beers using page offsets and limits.

    - Get beer details by id.

- Search for beer by name, description, tagline, food pairings, and price.

- Create a list of products on the main page.

- Create a search bar to filter products.

- Jump to the description page when the user clicks on a product.

- (Optional) Slicer to filter products by price.

- Create a shopping cart.

    Add products to the cart.

    Remove products from the cart.

    Calculate the total price of the products in the cart.

'
# AuthorRole.ASSISTANT - DevOpsGenArch: 'Sure, let's start by outlining the project structure in markdown format.

## Project Structure

```
shopp